# Demo — ¿Workflow o agente? Cuánta autonomía necesitas

> Demo que acompaña a [Patrones de diseño de
> agentes](../bloque2/02f-patrones-agentes.qmd#sec-workflows-vs-agentes)
> y prepara el [HPD 3
> (LangGraph)](../bloque2/hpd3-workflows-langgraph.qmd). Se ejecuta en
> el navegador y no necesita clave de API.

> **Las ejecuciones son simuladas**
>
> No hay ningún LLM detrás: cada recorrido está escrito a mano para
> ilustrar cómo se comporta cada patrón. Las cifras de llamadas y
> tiempos son orientativas (unos 1,5 s por llamada al LLM).

Una tienda online recibe correos de clientes y quiere responderlos con
un LLM. Hay muchas formas de montar el sistema, desde una sola llamada
hasta un agente que decide por sí mismo qué hacer. **Más autonomía no
siempre es mejor**: cuesta más llamadas, más tiempo y es menos
predecible.

Elige un correo y un patrón. El diagrama marca el camino que sigue el
correo; abajo, la tabla compara los cinco patrones con ese mismo correo.

``` shinylive-python
#| '!! shinylive warning !!': |
#|   shinylive does not work in self-contained HTML documents.
#|   Please set `embed-resources: false` in your metadata.
#| standalone: true
#| viewerHeight: 1380

import random

from shiny import reactive
from shiny.express import input, ui, render

BLUE, DARK, YELLOW, ORANGE, GREEN, RED, GRAY = (
    "#4B8BBE", "#306998", "#FFD43B", "#e65100", "#2e7d32", "#c62828", "#9ca3af")

# ── Los correos ──────────────────────────────────────────────────────────────
EMAILS = [
    {"tag": "Pregunta sencilla", "from": "Lucía",
     "text": "Hola, ¿a qué hora abre la tienda de Sevilla los sábados?"},
    {"tag": "Devolución", "from": "Marcos",
     "text": "Quiero devolver las zapatillas del pedido 4417, me quedan pequeñas."},
    {"tag": "Dos problemas a la vez", "from": "Elena",
     "text": "El pedido 5120 llegó con la pantalla rota y además me habéis cobrado dos veces. ¿Qué hago?"},
]

# ── Los patrones, de menos a más autonomía ───────────────────────────────────
# Nodos: id -> (x, y, etiqueta, tipo). Tipos: io, llm, code, tool.
PATTERNS = {
    "cadena": {
        "name": "Cadena simple", "family": "workflow",
        "desc": "Una sola llamada al LLM: «redacta una respuesta a este correo».",
        "nodes": {"in": (60, 110, "Correo", "io"), "llm": (320, 110, "LLM redacta", "llm"),
                  "out": (580, 110, "Respuesta", "io")},
        "edges": [("in", "llm"), ("llm", "out")],
    },
    "encadenado": {
        "name": "Encadenado", "family": "workflow",
        "desc": "Pasos fijos: extraer datos, consultar el pedido, redactar y comprobar la política.", "w": 44,
        "nodes": {"in": (45, 110, "Correo", "io"), "ext": (150, 110, "Extraer datos", "llm"),
                  "db": (260, 110, "Consultar pedido", "code"), "red": (370, 110, "Redactar", "llm"),
                  "pol": (480, 110, "¿Cumple política?", "code"), "out": (590, 110, "Respuesta", "io")},
        "edges": [("in", "ext"), ("ext", "db"), ("db", "red"), ("red", "pol"), ("pol", "out")],
    },
    "enrutado": {
        "name": "Enrutado", "family": "workflow",
        "desc": "Un clasificador elige UNA rama especializada según el tipo de correo.",
        "nodes": {"in": (60, 110, "Correo", "io"), "cls": (190, 110, "Clasificar", "llm"),
                  "inf": (360, 40, "Información", "llm"), "dev": (360, 110, "Devoluciones", "llm"),
                  "inc": (360, 180, "Incidencias", "llm"), "out": (560, 110, "Respuesta", "io")},
        "edges": [("in", "cls"), ("cls", "inf"), ("cls", "dev"), ("cls", "inc"),
                  ("inf", "out"), ("dev", "out"), ("inc", "out")],
    },
    "paralelo": {
        "name": "Paralelización", "family": "workflow",
        "desc": "Tres revisores revisan el correo a la vez (envíos, cobros, devoluciones) y un agregador une lo que encuentran.",
        "nodes": {"in": (60, 110, "Correo", "io"), "env": (230, 40, "Revisor envíos", "llm"),
                  "cob": (230, 110, "Revisor cobros", "llm"), "rdv": (230, 180, "Revisor devoluciones", "llm"),
                  "agg": (420, 110, "Agregador", "llm"), "out": (580, 110, "Respuesta", "io")},
        "edges": [("in", "env"), ("in", "cob"), ("in", "rdv"), ("env", "agg"), ("cob", "agg"),
                  ("rdv", "agg"), ("agg", "out")],
    },
    "agente": {
        "name": "Agente", "family": "agente",
        "desc": "El LLM decide en un bucle qué herramienta usar y cuándo parar.",
        "nodes": {"in": (60, 110, "Correo", "io"), "llm": (260, 110, "LLM decide", "llm"),
                  "tools": (440, 40, "Herramientas", "tool"), "out": (580, 110, "Respuesta", "io")},
        "edges": [("in", "llm"), ("llm", "tools"), ("tools", "llm"), ("llm", "out")],
    },
}
ORDER = list(PATTERNS)

# ── Qué pasa con cada correo ─────────────────────────────────────────────────
# result: ok | partial | fail. trace: [(nodo, texto)]. calls: llamadas al LLM. secs: tiempo aprox.
OK, PARTIAL, FAIL = "ok", "partial", "fail"
CASES = {
    ("cadena", 0): dict(calls=1, secs=1.5, result=OK, trace=[
        ("in", "Llega el correo."),
        ("llm", "El LLM tiene el horario en sus instrucciones y redacta la respuesta."),
        ("out", "«Los sábados abrimos de 10:00 a 14:00. ¡Te esperamos!»")],
        note="Basta y sobra: una pregunta de un paso no necesita más."),
    ("cadena", 1): dict(calls=1, secs=1.5, result=PARTIAL, trace=[
        ("in", "Llega el correo."),
        ("llm", "El LLM redacta sin poder consultar el pedido: no sabe si está dentro de plazo."),
        ("out", "«Claro, puedes devolverlas. Te enviaremos instrucciones.»")],
        note="Suena bien, pero no comprueba el pedido ni genera la devolución: promete algo que nadie ha verificado."),
    ("cadena", 2): dict(calls=1, secs=1.5, result=FAIL, trace=[
        ("in", "Llega el correo."),
        ("llm", "Sin datos ni herramientas, solo puede disculparse."),
        ("out", "«Sentimos mucho las molestias. Revisaremos tu caso.»")],
        note="No resuelve ninguno de los dos problemas."),

    ("encadenado", 0): dict(calls=2, secs=3.2, result=OK, trace=[
        ("in", "Llega el correo."), ("ext", "Extrae datos: no hay número de pedido."),
        ("db", "No hay pedido que consultar: se salta."), ("red", "Redacta con el horario."),
        ("pol", "No hay nada que comprobar."), ("out", "«Los sábados abrimos de 10:00 a 14:00.»")],
        note="Funciona, pero con pasos que no hacían falta: el doble de llamadas que la cadena simple."),
    ("encadenado", 1): dict(calls=2, secs=3.4, result=OK, trace=[
        ("in", "Llega el correo."), ("ext", "Extrae: pedido 4417, motivo «talla»."),
        ("db", "Pedido 4417: entregado hace 12 días."), ("red", "Redacta la respuesta con los datos reales."),
        ("pol", "Plazo de devolución: 30 días. 12 < 30 ✔. Genera la etiqueta DEV-4417."),
        ("out", "«Tu pedido está dentro de plazo. Aquí tienes la etiqueta DEV-4417.»")],
        note="El caso ideal para un workflow: la tarea tiene siempre los mismos pasos."),
    ("encadenado", 2): dict(calls=2, secs=3.4, result=PARTIAL, trace=[
        ("in", "Llega el correo."), ("ext", "Extrae: pedido 5120, motivo «rotura»."),
        ("db", "Pedido 5120: entregado ayer."), ("red", "Redacta una devolución por rotura."),
        ("pol", "Dentro de plazo ✔."), ("out", "«Te enviamos la etiqueta para devolver la pantalla.»")],
        note="El flujo se diseñó para devoluciones: resuelve la rotura e ignora el doble cobro."),

    ("enrutado", 0): dict(calls=2, secs=3.0, result=OK, trace=[
        ("in", "Llega el correo."), ("cls", "Clasifica: «información»."),
        ("inf", "La rama de información responde con el horario."),
        ("out", "«Los sábados abrimos de 10:00 a 14:00.»")],
        note="Funciona. Útil si llegan muchos tipos de correo distintos."),
    ("enrutado", 1): dict(calls=3, secs=4.5, result=OK, trace=[
        ("in", "Llega el correo."), ("cls", "Clasifica: «devolución»."),
        ("dev", "La rama de devoluciones consulta el pedido (12 días, dentro de plazo) y genera DEV-4417."),
        ("out", "«Tu pedido está dentro de plazo. Aquí tienes la etiqueta DEV-4417.»")],
        note="Funciona: la rama especializada es, por dentro, un encadenado."),
    ("enrutado", 2): dict(calls=3, secs=4.5, result=PARTIAL, trace=[
        ("in", "Llega el correo."), ("cls", "Clasifica: «incidencia» (tiene que elegir una sola categoría)."),
        ("inc", "La rama de incidencias abre un parte por la pantalla rota."),
        ("out", "«Hemos abierto la incidencia INC-882 por la rotura.»")],
        note="El enrutado manda cada correo por UNA rama: el doble cobro se queda sin atender."),

    ("paralelo", 0): dict(calls=4, secs=3.0, result=OK, trace=[
        ("in", "Llega el correo y se envía a los tres revisores a la vez."),
        ("env", "Envíos: no aplica."), ("cob", "Cobros: no aplica."), ("rdv", "Devoluciones: no aplica."),
        ("agg", "El agregador no encuentra problemas y responde con el horario."),
        ("out", "«Los sábados abrimos de 10:00 a 14:00.»")],
        note="Funciona, pero tres de las cuatro llamadas no aportan nada: coste desperdiciado."),
    ("paralelo", 1): dict(calls=4, secs=3.2, result=OK, trace=[
        ("in", "Llega el correo y se envía a los tres revisores a la vez."),
        ("env", "Envíos: no aplica."), ("cob", "Cobros: no aplica."),
        ("rdv", "Devoluciones: pedido 4417 dentro de plazo, genera DEV-4417."),
        ("agg", "El agregador redacta la respuesta."),
        ("out", "«Tu pedido está dentro de plazo. Aquí tienes la etiqueta DEV-4417.»")],
        note="Funciona, pero cuesta el doble que el encadenado para el mismo resultado."),
    ("paralelo", 2): dict(calls=4, secs=3.2, result=OK, trace=[
        ("in", "Llega el correo y se envía a los tres revisores a la vez."),
        ("env", "Envíos: pantalla rota en el pedido 5120 → abre la incidencia INC-882."),
        ("cob", "Cobros: dos cargos de 249 € el mismo día → solicita el reembolso de uno."),
        ("rdv", "Devoluciones: no aplica (se sustituye, no se devuelve)."),
        ("agg", "El agregador junta los dos hallazgos en una sola respuesta."),
        ("out", "«Te enviamos una pantalla nueva (INC-882) y te devolvemos el cargo duplicado de 249 €.»")],
        note="Resuelve los dos problemas SIN agente: porque se diseñó un revisor para cada tipo de problema."),
}

# El agente no hace siempre lo mismo: varias ejecuciones posibles por correo.
AGENT_RUNS = {
    0: [
        dict(result=OK, tools=["horario_tienda"],
             answer="«Los sábados abrimos de 10:00 a 14:00.»"),
        dict(result=OK, tools=["buscar_tienda", "horario_tienda"],
             answer="«La tienda de Sevilla abre los sábados de 10:00 a 14:00.»"),
        dict(result=OK, tools=["horario_tienda"],
             answer="«Los sábados abrimos de 10:00 a 14:00.»"),
    ],
    1: [
        dict(result=OK, tools=["consultar_pedido", "crear_devolucion"],
             answer="«Tu pedido está dentro de plazo. Aquí tienes la etiqueta DEV-4417.»"),
        dict(result=OK, tools=["consultar_pedido", "politica_devoluciones", "crear_devolucion"],
             answer="«Tu pedido está dentro de plazo. Aquí tienes la etiqueta DEV-4417.»"),
        dict(result=PARTIAL, tools=["consultar_pedido"],
             answer="«Tu pedido está dentro de plazo, puedes devolverlo.» (no llega a crear la devolución)"),
    ],
    2: [
        dict(calls=6, secs=9.4, result=OK,
             tools=["consultar_pedido", "abrir_incidencia", "consultar_pagos", "solicitar_reembolso"],
             answer="«Te enviamos una pantalla nueva (INC-882) y te devolvemos el cargo duplicado de 249 €.»"),
        dict(calls=8, secs=12.5, result=OK,
             tools=["consultar_pedido", "consultar_pagos", "abrir_incidencia", "consultar_pagos",
                    "politica_reembolsos", "solicitar_reembolso"],
             answer="«Te enviamos una pantalla nueva (INC-882) y te devolvemos el cargo duplicado de 249 €.»"),
        dict(result=PARTIAL, tools=["consultar_pedido", "abrir_incidencia"],
             answer="«Hemos abierto la incidencia INC-882 por la rotura.» (se olvida del doble cobro)"),
    ],
}

RESULT = {OK: ("✔ Resuelto", GREEN), PARTIAL: ("⚠ A medias", ORANGE), FAIL: ("✘ No resuelve", RED)}
KIND = {"io": (BLUE, "#fff"), "llm": (DARK, "#fff"), "code": (YELLOW, "#000"), "tool": (ORANGE, "#fff")}


def case(p, e, run=0):
    """Ejecución de un patrón sobre un correo (para el agente, la variante `run`)."""
    if p != "agente":
        return CASES[(p, e)]
    r = AGENT_RUNS[e][run]
    calls = len(r["tools"]) + 1  # una decisión por herramienta y otra para responder
    trace = [("in", "Llega el correo.")]
    for t in r["tools"]:
        trace += [("llm", f"Decide usar <code>{t}</code>."), ("tools", f"Ejecuta <code>{t}</code> y devuelve el resultado.")]
    trace += [("llm", "Decide que ya puede responder."), ("out", r["answer"])]
    notes = {OK: "Lo resuelve, decidiendo él mismo qué consultar.",
             PARTIAL: "Esta vez se para antes de tiempo: el agente decide cuándo ha terminado, y a veces se equivoca."}
    return dict(calls=calls, secs=1.55 * calls, result=r["result"], trace=trace, note=notes[r["result"]],
                loops=len(r["tools"]))


# ── Interfaz ─────────────────────────────────────────────────────────────────
ui.tags.style("""
body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif}
.card{border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;margin:10px 0;background:#fff}
.muted{color:#6b7280;font-size:.85em}
.emails{display:grid;grid-template-columns:repeat(3,1fr);gap:8px}
.email{border:1.5px solid #e5e7eb;border-radius:10px;padding:8px 10px;cursor:pointer;font-size:.86em;background:#fff}
.email:hover{border-color:#4B8BBE}.email.on{border-color:#306998;box-shadow:0 0 0 3px #30699833}
.email .t{font-weight:700;font-size:.8em;color:#306998;text-transform:uppercase;letter-spacing:.03em}
.pats{display:flex;gap:6px;flex-wrap:wrap;align-items:center}
.pat{border:1.5px solid #306998;color:#306998;border-radius:999px;padding:4px 14px;cursor:pointer;font-size:.9em;background:#fff}
.pat:hover{background:#e8f0f8}.pat.on{background:#306998;color:#fff}
.pat.ag{border-color:#e65100;color:#e65100}.pat.ag.on{background:#e65100;color:#fff}
.axis{font-size:.75em;color:#6b7280;margin-top:3px}
.tiles{display:flex;gap:8px;flex-wrap:wrap}
.tile{flex:1;min-width:110px;border:1px solid #e5e7eb;border-radius:8px;padding:4px 10px}
.tile b{display:block;font-size:1.35em}
.trace{margin:0;padding-left:0;list-style:none;font-size:.88em}
.trace li{display:flex;gap:8px;align-items:flex-start;margin:3px 0}
.num{flex:none;width:20px;height:20px;border-radius:50%;background:#306998;color:#fff;font-size:.75em;
  display:flex;align-items:center;justify-content:center;font-weight:700;margin-top:1px}
.cmp{width:100%;font-size:.86em;border-collapse:collapse}
.cmp td,.cmp th{padding:5px 6px;border-bottom:1px solid #f3f4f6;text-align:left}
.cmp tr.on{background:#eef4fa}.cmp tr{cursor:pointer}
.star{background:#2e7d32;color:#fff;border-radius:4px;padding:0 6px;font-size:.8em;font-weight:600}
.runs{display:flex;gap:8px;flex-wrap:wrap;margin-top:6px}
.run{border-radius:8px;padding:4px 10px;font-size:.82em;color:#fff}
@keyframes pop{from{transform:scale(.6);opacity:0}to{transform:scale(1);opacity:1}}
.run{animation:pop .3s ease-out both}
@media (max-width:620px){.emails{grid-template-columns:1fr}}
""")

email = reactive.value(1)
pattern = reactive.value("encadenado")
runs = reactive.value([])
agent_run = reactive.value(0)


@reactive.effect
@reactive.event(input.pick_email)
def _pe():
    email.set(int(input.pick_email()))
    runs.set([])
    agent_run.set(0)


@reactive.effect
@reactive.event(input.pick_pat)
def _pp():
    pattern.set(input.pick_pat())
    runs.set([])
    agent_run.set(0)


@reactive.effect
@reactive.event(input.run5)
def _run5():
    p, e = pattern(), email()
    if p == "agente":
        idx = [random.randrange(len(AGENT_RUNS[e])) for _ in range(5)]
        agent_run.set(idx[-1])
        runs.set([case(p, e, i) for i in idx])
    else:
        runs.set([case(p, e)] * 5)


def onclick(name, value):
    return f"onclick=\"Shiny.setInputValue('{name}', '{value}', {{priority: 'event'}})\""


@render.ui
def pickers():
    e, p = email(), pattern()
    cards = "".join(
        f"<div class='email{' on' if i == e else ''}' {onclick('pick_email', i)}>"
        f"<div class='t'>{m['tag']}</div><b>{m['from']}:</b> {m['text']}</div>"
        for i, m in enumerate(EMAILS)
    )
    pats = "".join(
        f"<span class='pat{' ag' if PATTERNS[k]['family'] == 'agente' else ''}{' on' if k == p else ''}' "
        f"{onclick('pick_pat', k)}>{PATTERNS[k]['name']}</span>"
        for k in ORDER
    )
    return ui.HTML(
        f"<div style='font-weight:600;margin-bottom:4px'>1. Elige un correo de cliente</div>"
        f"<div class='emails'>{cards}</div>"
        f"<div style='font-weight:600;margin:12px 0 4px'>2. Elige cómo lo procesa el sistema</div>"
        f"<div class='pats'>{pats}</div>"
        f"<div class='axis'>← el código decide el orden (workflow) · · · el LLM decide qué hacer (agente) →</div>"
    )


def graph(p, path, loops):
    P = PATTERNS[p]
    nodes, visited = P["nodes"], set(path)
    order = {}
    for i, n in enumerate(path):
        order.setdefault(n, i + 1)
    parts = ["<defs><marker id='ar' viewBox='0 0 10 10' refX='9' refY='5' markerWidth='6' markerHeight='6' "
             "orient='auto'><path d='M0,0 L10,5 L0,10 z' fill='#6b7280'/></marker>"
             "<marker id='arh' viewBox='0 0 10 10' refX='9' refY='5' markerWidth='6' markerHeight='6' "
             "orient='auto'><path d='M0,0 L10,5 L0,10 z' fill='#e65100'/></marker></defs>"]
    used = {(path[i], path[i + 1]) for i in range(len(path) - 1)}
    if p == "paralelo":
        used |= {("in", n) for n in ("env", "cob", "rdv")} | {(n, "agg") for n in ("env", "cob", "rdv")}
    W = P.get("w", 52)
    for a, b in P["edges"]:
        (x1, y1, *_), (x2, y2, *_) = nodes[a], nodes[b]
        on = (a, b) in used
        dy = 0
        if p == "agente" and {a, b} == {"llm", "tools"}:
            dy = -8 if a == "llm" else 8
        sx, sy = x1 + (W if x2 > x1 else -W if x2 < x1 else 0), y1 + dy
        ex, ey = x2 - (W if x2 > x1 else -W if x2 < x1 else 0), y2 + dy
        parts.append(
            f"<line x1='{sx}' y1='{sy}' x2='{ex}' y2='{ey}' stroke='{ORANGE if on else '#d1d5db'}' "
            f"stroke-width='{2.6 if on else 1.4}' marker-end='url(#{'arh' if on else 'ar'})'/>"
        )
    for n, (x, y, label, kind) in nodes.items():
        fill, tc = KIND[kind]
        on = n in visited
        words = label.split(" ")
        lines = [" ".join(words[: (len(words) + 1) // 2]), " ".join(words[(len(words) + 1) // 2:])] \
            if len(label) > 13 else [label]
        text = "".join(
            f"<tspan x='{x}' dy='{0 if i == 0 else 13}'>{l}</tspan>" for i, l in enumerate(lines))
        ty = y + 4 - 6 * (len(lines) - 1)
        parts.append(
            f"<g opacity='{1 if on else 0.3}'><rect x='{x - W}' y='{y - 22}' width='{2 * W}' height='44' rx='9' "
            f"fill='{fill}' stroke='{'#111' if on else 'none'}' stroke-width='1.5'/>"
            f"<text x='{x}' y='{ty}' text-anchor='middle' font-size='12' font-weight='600' fill='{tc}'>{text}</text></g>"
        )
        if n in order:
            parts.append(
                f"<circle cx='{x + W - 4}' cy='{y - 22}' r='9' fill='#fff' stroke='#111'/>"
                f"<text x='{x + W - 4}' y='{y - 18}' text-anchor='middle' font-size='10' font-weight='700'>{order[n]}</text>"
            )
    if p == "agente" and loops:
        parts.append(
            f"<text x='440' y='8' text-anchor='middle' font-size='12' font-weight='700' fill='{ORANGE}'>"
            f"↻ {loops} vuelta{'s' if loops != 1 else ''}</text>")
    ys = [y for (_, y, *_) in nodes.values()]
    top, bottom = min(ys) - 34, max(ys) + 30
    if p == "agente":
        top = -8
    legend = "".join(
        f"<rect x='{20 + i * 150}' y='{bottom + 4}' width='12' height='12' rx='3' fill='{KIND[k][0]}'/>"
        f"<text x='{38 + i * 150}' y='{bottom + 14}' font-size='11' fill='#374151'>{t}</text>"
        for i, (k, t) in enumerate([("llm", "llamada al LLM"), ("code", "código normal"),
                                    ("tool", "herramientas"), ("io", "entrada / salida")])
    )
    return (f"<svg viewBox='0 {top} 640 {bottom - top + 22}' width='100%' "
            f"style='display:block;max-width:680px;margin:6px auto' "
            f"font-family='system-ui,sans-serif'>{''.join(parts)}{legend}</svg>")


@render.ui
def run_view():
    p, e = pattern(), email()
    c = case(p, e, agent_run())
    path = [n for n, _ in c["trace"]]
    label, color = RESULT[c["result"]]
    steps = "".join(
        f"<li><span class='num' style='background:{KIND[PATTERNS[p]['nodes'][n][3]][0]};"
        f"color:{KIND[PATTERNS[p]['nodes'][n][3]][1]}'>{i + 1}</span><span>"
        f"<b>{PATTERNS[p]['nodes'][n][2]}:</b> {t}</span></li>"
        for i, (n, t) in enumerate(c["trace"])
    )
    tiles = (
        f"<div class='tiles'>"
        f"<div class='tile'><span class='muted'>Llamadas al LLM</span><b>{c['calls']}</b></div>"
        f"<div class='tile'><span class='muted'>Tiempo aprox.</span><b>{c['secs']:.1f} s</b></div>"
        f"<div class='tile'><span class='muted'>Resultado</span><b style='color:{color}'>{label}</b></div></div>"
    )
    return ui.HTML(
        f"<div class='card'><div><b>{PATTERNS[p]['name']}</b> · <span class='muted'>{PATTERNS[p]['desc']}</span></div>"
        f"{graph(p, path, c.get('loops', 0))}{tiles}"
        f"<ol class='trace' style='margin-top:8px'>{steps}</ol>"
        f"<p style='border-left:4px solid {color};padding-left:10px;margin:8px 0 0'>{c['note']}</p></div>"
    )


ui.input_action_button("run5", "Ejecutar 5 veces el mismo correo", class_="btn-outline-primary btn-sm")


@render.ui
def runs_view():
    rs = runs()
    if not rs:
        return ui.HTML("<p class='muted' style='margin:4px 0'>¿Hace siempre lo mismo? Pulsa el botón y compruébalo.</p>")
    chips = "".join(
        f"<span class='run' style='background:{RESULT[r['result']][1]};animation-delay:{0.08 * i:.2f}s'>"
        f"#{i + 1}: {r['calls']} llamadas · {RESULT[r['result']][0]}</span>"
        for i, r in enumerate(rs)
    )
    calls = [r["calls"] for r in rs]
    same = len({(r["calls"], r["result"]) for r in rs}) == 1
    msg = ("Las cinco ejecuciones son idénticas: el código decide el camino, así que es predecible y fácil de probar."
           if same else
           f"Entre {min(calls)} y {max(calls)} llamadas, y no siempre con el mismo resultado: "
           f"el LLM decide el camino en cada ejecución. Más difícil de probar y de presupuestar.")
    return ui.HTML(f"<div class='runs'>{chips}</div><p class='muted' style='margin:6px 0 0'>{msg}</p>")


@render.ui
def compare():
    e, p = email(), pattern()
    rows, simplest = [], None
    for k in ORDER:
        if k == "agente":
            rs = AGENT_RUNS[e]
            n = [len(r["tools"]) + 1 for r in rs]
            calls = f"{min(n)}–{max(n)}"
            results = {r["result"] for r in rs}
            res = "✔ casi siempre" if OK in results and len(results) > 1 else RESULT[rs[0]["result"]][0]
            color = ORANGE if len(results) > 1 else RESULT[rs[0]["result"]][1]
            pred = "No"
            ok = OK in results
        else:
            c = CASES[(k, e)]
            calls, (res, color), pred, ok = str(c["calls"]), RESULT[c["result"]], "Sí", c["result"] == OK
        star = ""
        if ok and simplest is None and k != "agente":
            simplest = k
            star = " <span class='star'>la más simple que lo resuelve</span>"
        rows.append(
            f"<tr class='{'on' if k == p else ''}' {onclick('pick_pat', k)}><td><b>{PATTERNS[k]['name']}</b>{star}</td>"
            f"<td>{calls}</td><td style='color:{color};font-weight:600'>{res}</td><td>{pred}</td></tr>"
        )
    return ui.HTML(
        f"<div class='card'><b>Todos los patrones con este correo</b> "
        f"<span class='muted'>(pulsa una fila para verla)</span>"
        f"<table class='cmp' style='margin-top:6px'><thead><tr><th>Patrón</th><th>Llamadas</th>"
        f"<th>Resultado</th><th>¿Siempre igual?</th></tr></thead><tbody>{''.join(rows)}</tbody></table></div>"
    )
```

## Qué observar

1.  **Correo de Lucía (pregunta sencilla).** La cadena simple lo
    resuelve con una llamada. Todo lo demás también funciona, pero gasta
    de dos a cuatro veces más. La tabla marca la cadena simple como «la
    más simple que lo resuelve».
2.  **Correo de Marcos (devolución).** La cadena simple suena bien, pero
    no comprueba el pedido: promete algo que nadie ha verificado. El
    **encadenado** es el caso ideal: siempre los mismos pasos, con los
    datos reales del pedido.
3.  **Correo de Elena (dos problemas).** El encadenado y el enrutado
    resuelven solo uno: están diseñados para un tipo de problema por
    correo. La **paralelización** resuelve los dos *sin agente*, porque
    tiene un revisor para cada tipo de problema. El agente también lo
    resuelve.
4.  Con el **agente** seleccionado, pulsa **«Ejecutar 5 veces»**.
    Cambian las llamadas y, a veces, el resultado: el agente decide
    cuándo ha terminado y en ocasiones se para antes de tiempo. Repite
    con un workflow: cinco ejecuciones idénticas.

## Qué has visto

- **Empieza por lo más simple que funcione.** La tabla de cada correo
  responde a la pregunta «¿hace falta un agente?». En dos de los tres
  correos, no.
- Los **workflows** (el código decide el orden) son predecibles, baratos
  y fáciles de probar. Funcionan cuando conoces la estructura de la
  tarea, y fallan cuando llega algo para lo que no se diseñaron.
- Un **agente** se adapta a lo imprevisto, pero cada ejecución es
  distinta: más difícil de probar, de depurar y de presupuestar.
- A menudo un workflow bien diseñado (aquí, la paralelización) cubre el
  caso difícil sin ceder el control al LLM.

> **Para el HPD 3**
>
> En LangGraph, cada uno de estos diagramas es un grafo: los nodos son
> las cajas y las aristas condicionales son las decisiones. La
> diferencia entre un workflow y un agente está en **quién decide qué
> arista seguir**: tu código o el LLM.

Para la descripción completa de cada patrón, vuelve a [Patrones de
diseño de
agentes](../bloque2/02f-patrones-agentes.qmd#sec-workflow-patterns).